Remove previous tensorflow and install spefific version

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

Create model

In [15]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models


# [1] Create depthwise_conv2d, without activation function
model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(28, 28, 1)),
    tf.keras.layers.DepthwiseConv2D(kernel_size=(3, 3), activation=None, use_bias=True)
])


# [2] Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# [3] Load MNIST dataset
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()

# Normalize the data to [0, 1] range
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255
# Select only the first image from x_train and corresponding label
single_image_train = x_train[0:1]
single_label_train = y_train[0:1]

# Reshape to fit Conv2D input (28, 28, 1)
single_image_train = single_image_train.reshape(1, 28, 28, 1)

# Train the model with the single image
model.fit(single_image_train, single_label_train.astype('float32'), epochs=5, batch_size=1)

# Evaluate the model with a single test image
single_image_test = x_test[0:1].reshape(1, 28, 28, 1)
model.evaluate(single_image_test, y_test[0:1].astype('float32'))

# Summarize the model structure
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 536ms/step - loss: 25.4395
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 25.4146
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 25.3897
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 25.3649
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 25.3400
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - loss: 49.2455


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ depthwise_conv2d_1              │ (None, 26, 26, 1)      │            10 │
│ (DepthwiseConv2D)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32 (132.00 B)

 Trainable params: 10 (40.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 22 (92.00 B)

Lượng tử hóa xuống 8bit

In [16]:
def representative_data_gen():
  for input_value in single_image_train:
    # The input value needs to be a 4D array (batch, height, width, channels)
    # even if batch size is 1
    yield [input_value.reshape(1, 28, 28, 1)]

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

converter.representative_dataset = representative_data_gen
tflite_model = converter.convert()

# Save the model
with open('model_quantized.tflite', 'wb') as f:
    f.write(tflite_model)

In [25]:
model.save('my_model.keras')
# Convert the model to TFLite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the TFLite model to a file
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpfq62s7v3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor_2')
Output Type:
  TensorSpec(shape=(None, 26, 26, 1), dtype=tf.float32, name=None)
Captures:
  138946942835344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138946942835920: TensorSpec(shape=(), dtype=tf.resource, name=None)
